In [10]:
import pandas as pd
from collections import defaultdict
import statbotics
import requests
import json
import os
from pathlib import Path

match_json_path = Path("betterSB/orwilmatchmath.json")
match_data = json.loads(match_json_path.read_text())

sb = statbotics.Statbotics()
try:
    score_sd = sb.get_year(2025, fields=["score_sd"])
except Exception:
    score_sd = {"score_sd": None}

k = -5/8
print(score_sd)


{'score_sd': 36.84}


In [11]:
def calc_win_odds(k, score_sd_value, red_score, blue_score):
    norm_diff = (red_score - blue_score) / score_sd_value if score_sd_value else 0.0
    odds = 1 / (1 + 10**(k * norm_diff))
    return odds

def get_team_total_points(team):
    if not team:
        return 0.0
    points = team.get("points", {})
    metric = team.get("selected_metric")
    if metric:
        total_points = points.get(metric, {}).get("total_points")
        if total_points is not None:
            return float(total_points)
    fallback = team.get("selected_total")
    if fallback is None:
        fallback = team.get("selected_value", 0.0)
    try:
        return float(fallback)
    except (TypeError, ValueError):
        return 0.0

def get_alliance_team_keys(match, alliance):
    return [team.get("team_key") for team in match.get("teams", []) if team.get("alliance") == alliance]

def fetch_tba_scores(match_key):
    url = f"https://www.thebluealliance.com/api/v3/match/{match_key}"
    headers = {"X-TBA-Auth-Key": "uqTThWSrIgK7D7M3ct9fnwfIrj9m7ZzuCjwsgWsHzMtRl2xRNIm8pEQXVhfwOsBv"}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return None, None
        data = resp.json()
        alliances = data.get("alliances") or {}
        red = alliances.get("red") or {}
        blue = alliances.get("blue") or {}
        return red.get("score"), blue.get("score")
    except Exception:
        return None, None

def team_prior_average(prior_selected, team_key):
    if not team_key:
        return 0.0
    values = prior_selected.get(team_key, [])
    if not values:
        return 0.0
    return sum(values) / len(values)


In [12]:
sorted_matches = sorted(
    match_data.get("matches", []),
    key=lambda match: (
        match.get("set_number", 0),
        match.get("match_number", 0),
        match.get("match_key", ""),
    ),
)


In [13]:
score_sd_value = score_sd.get("score_sd")
processed_matches = set()
prior_selected = defaultdict(list)

def describe_team_prior(team_key):
    values = prior_selected.get(team_key, [])
    avg = team_prior_average(prior_selected, team_key)
    return values, avg
rows = []
logged = False
for match in sorted_matches:
    match_key = match.get("match_key")
    if not match_key or match_key in processed_matches:
        continue
    processed_matches.add(match_key)

    red_team_keys = get_alliance_team_keys(match, "red")[:3]
    blue_team_keys = get_alliance_team_keys(match, "blue")[:3]

    red_score = sum(team_prior_average(prior_selected, team) for team in red_team_keys)
    blue_score = sum(team_prior_average(prior_selected, team) for team in blue_team_keys)

    odds = calc_win_odds(k, score_sd_value, red_score, blue_score)

    if match.get("match_number") == 29 and not logged:
        logged = True
        print("Match 29 predicted averages per team:")
        for label, keys in [("red", red_team_keys), ("blue", blue_team_keys)]:
            print(f"  {label.capitalize()} alliance teams: {keys}")
            for team_key in keys:
                values, avg = describe_team_prior(team_key)
                print(f"    {team_key}: values={values}, average={avg:.2f}")
        print("  Totals: red_score=", red_score, "blue_score=", blue_score)

    r1_key = red_team_keys[0] if len(red_team_keys) > 0 else None
    r2_key = red_team_keys[1] if len(red_team_keys) > 1 else None
    r3_key = red_team_keys[2] if len(red_team_keys) > 2 else None
    b1_key = blue_team_keys[0] if len(blue_team_keys) > 0 else None
    b2_key = blue_team_keys[1] if len(blue_team_keys) > 1 else None
    b3_key = blue_team_keys[2] if len(blue_team_keys) > 2 else None

    ba_red_score, ba_blue_score = fetch_tba_scores(match_key)
    if ba_red_score is None or ba_blue_score is None:
        correct_pred = None
    else:
        actual_red_win = ba_red_score > ba_blue_score
        predicted_red_win = odds >= 0.5
        correct_pred = int(actual_red_win == predicted_red_win)

    rows.append({
        "match_number": int(match.get("match_number", 0)),
        "r1": r1_key,
        "r2": r2_key,
        "r3": r3_key,
        "b1": b1_key,
        "b2": b2_key,
        "b3": b3_key,
        "red_score": red_score,
        "blue_score": blue_score,
        "red_win_odds": odds,
        "ba_redScore": ba_red_score,
        "ba_blueScore": ba_blue_score,
        "correct_pred": correct_pred,
    })

    team_lookup = {
        team.get("team_key"): team for team in match.get("teams", []) if team.get("team_key")
    }

    for team_key in red_team_keys + blue_team_keys:
        value = get_team_total_points(team_lookup.get(team_key))
        prior_selected[team_key].append(value)

preds_df = pd.DataFrame(rows)
preds_df.to_csv("preds1.csv", index=False)
preds_df.head()


Match 29 predicted averages per team:
  Red alliance teams: ['frc997', 'frc4043', 'frc4125']
    frc997: values=[79.00000000000003, 78.91571662345935, 144.25243383999282, 55.2], average=89.34
    frc4043: values=[38.000000000000014, 9.941605839416027, -74.71996146450256, -52.343850541212454], average=-19.78
    frc4125: values=[36.33333333333334, 21.417675183027086, 80.09448987174068, -42.54666068068049], average=23.82
  Blue alliance teams: ['frc2733', 'frc5920', 'frc6465']
    frc2733: values=[12.66666666666667, -6.7736026945951116, -30.219503213390468, 182.68922096792357], average=39.59
    frc5920: values=[47.77, 37.10276867642949, 41.66666666666668, -15.727099029546212, 30.017737652459136, 42.806490072222346], average=30.61
    frc6465: values=[22.000000000000004, 33.50915154401915, 168.35171687722732, 72.34131234655243], average=74.05
  Totals: red_score= 93.38619550114346 blue_score= 144.24733462997278


,match_number,r1,r2,r3,b1,b2,b3,red_score,blue_score,red_win_odds,ba_redScore,ba_blueScore,correct_pred
0,1,frc5920,frc1540,frc5468,frc2811,frc9023,frc955,0.000000,0.000000,0.500000,329,253,1
1,1,frc7034,frc10991,frc9438,frc2521,frc4125,frc6845,0.000000,0.000000,0.500000,24,109,0
2,1,frc3712,frc1540,frc5468,frc4692,frc1359,frc8532,283.204731,0.000000,0.999984,289,59,1
3,2,frc5920,frc1540,frc5468,frc2811,frc9023,frc955,327.923332,247.284238,0.958912,247,225,1
4,2,frc846,frc997,frc9430,frc4043,frc749,frc2926,0.000000,0.000000,0.500000,237,114,1
